# Comparing $G_{high}$ Validation Scores

Points show individual seeds; lines show their mean and error bars show **±1 sample standard deviation (S.D., ddof=1)** at each initial annotation budget. The 10% budget uses vermillion in place of blue.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import PercentFormatter

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 180, "savefig.dpi": 600,
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.titlesize": 13, "axes.titleweight": "semibold",
    "axes.labelsize": 11, "axes.edgecolor": "#555555",
    "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": "#DDDDDD", "grid.linewidth": 0.6,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

# Support execution from either the notebook directory or the repository root.
experiment_dir = Path.cwd()
if not (experiment_dir / "compare_g_high_val_scores.ipynb").exists():
    experiment_dir = experiment_dir / "experiments" / "1_determine_initial_pool"
runs_dir = experiment_dir / "runs"
if not runs_dir.is_dir():
    raise FileNotFoundError(f"Runs directory not found: {runs_dir.resolve()}")

records = []
missing_runs = []
for run_dir in sorted(path for path in runs_dir.iterdir() if path.is_dir()):
    score_path = run_dir / "round_000" / "checkpoint_val_g_high.json"
    if not score_path.exists():
        missing_runs.append(run_dir.name)
        continue
    with (run_dir / "config.json").open() as file:
        config = json.load(file)
    with score_path.open() as file:
        scores = json.load(file)
    records.append({
        "run": run_dir.name,
        "initial_pool_pct": float(config["initial_labeled_pool_size"]),
        "seed": int(config["seed"]),
        "val_f1_event": float(scores["val_f1_event"]),
        "val_f1_element": float(scores["val_f1_element"]),
        "val_edit": float(scores["val_edit"]),
    })

if not records:
    raise FileNotFoundError(f"No round_000/checkpoint_val_g_high.json files found under {runs_dir.resolve()}")
results_df = (pd.DataFrame.from_records(records)
              .sort_values(["initial_pool_pct", "seed"]).reset_index(drop=True))
print(f"Loaded {len(results_df)} runs across {results_df['initial_pool_pct'].nunique()} initial budgets")
if missing_runs:
    print(f"Skipped {len(missing_runs)} runs without G_high validation scores: {', '.join(missing_runs)}")


In [ ]:
# Okabe–Ito palette: blue for means, vermillion for the selected 10% budget.
BLUE = "#0072B2"
VERMILLION = "#D55E00"
metric_config = {
    "val_f1_event": ("(a) Event F1", "F1 score", (0, 1)),
    "val_f1_element": ("(b) Element F1", "F1 score", (0, 1)),
    "val_edit": ("(c) Edit score", "Edit score", (0, 100)),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharex=True)
summaries = {}
for ax, (metric, (title, ylabel, limits)) in zip(axes, metric_config.items()):
    # pandas std uses ddof=1: sample standard deviation across seeds.
    summary = results_df.groupby("initial_pool_pct", sort=True)[metric].agg(["mean", "std", "count"])
    summaries[metric] = summary
    ax.scatter(results_df["initial_pool_pct"], results_df[metric],
               color="#777777", alpha=0.35, s=18, linewidths=0,
               label="Individual seeds", zorder=2)
    ax.errorbar(summary.index.to_numpy(), summary["mean"],
                yerr=summary["std"], color=BLUE,
                marker="o", markerfacecolor="white", markeredgewidth=1.2,
                markersize=4.5, linewidth=1.8, elinewidth=1.2, capsize=3.5,
                label="Mean ± 1 S.D. across seeds", zorder=3)
    if 10.0 in summary.index:
        selected = summary.loc[10.0]
        ax.errorbar([10], [selected["mean"]], yerr=[selected["std"]],
                    fmt="o", color=VERMILLION, markerfacecolor=VERMILLION,
                    markeredgewidth=1.2, markersize=4.5, elinewidth=1.2,
                    capsize=3.5, zorder=4)
    # Actual percentages preserve the spacing between annotation budgets.
    ax.set(title=title, xlabel="Initial annotation budget", ylabel=ylabel,
           xscale="linear", xlim=(0, 103), ylim=limits)
    ax.set_xticks([0, 10, 25, 50, 75, 100])
    ax.xaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.7)
    ax.grid(axis="x", visible=False)
    ax.tick_params(axis="both", labelsize=10)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 0.005),
           ncol=2, frameon=False, fontsize=10)
fig.suptitle(r"$G_{high}$ validation performance", fontsize=15, fontweight="semibold", y=0.98)
fig.tight_layout(rect=(0, 0.10, 1, 0.94), w_pad=2.5)

plt.show()
